# Set-up

In [ ]:
### Imports ###

import pandas as pd
import pickle
from skfda import FDataGrid
from datetime import date, datetime, timedelta
import numpy as np

from src.preprocessing import GMEPreprocessor, ExogPreprocessor
from src.curves import SupplyDemandTimeSeries
from src.forecasting import LassoVARX, SupplyDemandForecaster
from src.utils import fix_daylight_saving_time

In [34]:
### Parameters ###

# Input paths
bids_path = 'data/source/MGPDomandaOfferta/MGPDomandaOfferta.pkl'
coupling_path = 'data/source/MGPMarketCoupling/balance_coupling.csv'
prices_path = 'data/source/MGPPrezzi/mgp_prices.pkl'
exog_path = 'data/source/predictors.pkl'

# Processed paths
curves_path = 'data/processed/sdts.pkl'

# Output paths
pred_curves_path = 'data/output/sdts_pred.pkl'
pred_prices_path = 'data/output/prices_pred.pkl'

# Whether to re-preprocess curves (long!)
rerun_curves_preprocessing = False

# Model parameters

K_supply = 5
K_demand = 3

exog_variables = [
    'GFSo Solar ITA',
    'ECo Wind ITA',
    'Load_IT',
    'FR > IT',
    'IT > FR',
    'CH > IT',
    'IT > CH',
    'AT > IT',
    'IT > AT',
    'SI > IT',
    'IT > SI',
    'IT > ME',
    'ME > IT',
    'IT > GR',
    'GR > IT',
]

# Calibration window and testing period
calibration_window = timedelta(days=358)
test_start_date = date(2024, 1, 1)
test_end_date = date(2024, 12, 31)
print(f"Calibration_window: {calibration_window.days} days")
print(f"Testing period: {test_start_date} to {test_end_date}")

Calibration_window: 358 days
Testing period: 2024-01-01 to 2024-12-31


In [ ]:
### Read data ###

bids = pd.read_pickle(bids_path)
coupling = pd.read_csv(coupling_path)
prices = pd.read_pickle(prices_path)
exog = pd.read_pickle(exog_path)

In [ ]:
### Preprocessing ###

# Start and end datetimes
test_start = pd.Timestamp(test_start_date) # Time information automatically set at 00:00:00
test_end = pd.Timestamp(test_end_date) + timedelta(hours=23) # Time information set to 23:00:00
train_start = test_start - calibration_window
preprocess_start = train_start - timedelta(weeks=1) # We need one week of past data to compute the lags

# Curves
if rerun_curves_preprocessing:
    preprocessor = GMEPreprocessor()
    data_matrix_off, grid_points = preprocessor.get_curves_dataset(bids, type='OFF', balance_df=coupling)
    data_matrix_bid, _ = preprocessor.get_curves_dataset(bids, type='BID', balance_df=coupling)

    sd = SupplyDemandTimeSeries(
        FDataGrid(data_matrix_off, grid_points, sample_names=preprocessor.timestamps, extrapolation='bounds'),
        FDataGrid(data_matrix_bid, grid_points, sample_names=preprocessor.timestamps, extrapolation='bounds')
    )
    sd.to_pickle(curves_path)
else:
    with open(curves_path, 'rb') as file:
        sd = pickle.load(file)
sd = sd[preprocess_start:test_end]

# Exog
exogprep = ExogPreprocessor(
    start_date=preprocess_start.date(),
    end_date=test_end_date,
    exog_variables=exog_variables
)
exog = exogprep.preprocess_exog(exog)

# Prices
prices = fix_daylight_saving_time(prices)
prices_true = prices.loc[test_start:test_end, 'NAT']

100%|██████████| 17712/17712 [02:08<00:00, 138.14it/s]


In [ ]:
### Forecasting ###

model = LassoVARX(ar_structure='concurrent', var_structure='concurrent',
                    calibration_window=calibration_window)

forecaster = SupplyDemandForecaster(model, exogprep, K_supply=K_supply, K_demand=K_demand)
sd_pred = forecaster.fit_forecast_daily_recal(sd, exog, test_start=date(2024, 1, 1))
prices_pred = sd_pred.get_clearing_prices()

Daily Recalibration Progress: 100%|██████████| 366/366 [13:28<00:00,  2.21s/it]


In [ ]:
### Evaluate and save ###

sd_pred.to_pickle(pred_curves_path)
prices_pred.to_pickle(pred_prices_path)
print("MAE: {:.2f}€/MWh".format((prices_true - prices_pred).abs().mean()))

2025-09-25 11:06:38,496 - INFO - SupplyDemandTimeSeries saved to data/output/sdts_pred.pkl


MAE 8.35€/MWh


<HR>

# Tests